In [2]:
import os
from dotenv import load_dotenv
from langchain_openrouter import ChatOpenRouter

load_dotenv(dotenv_path=".env", override=True)

api_key = os.getenv("OPENROUTER_API_KEY")
if not api_key:
    raise RuntimeError("OPENROUTER_API_KEY is missing from .env")

llm = ChatOpenRouter(
    model="deepseek/deepseek-v4-flash-0731",
    api_key=api_key,
    temperature=0,
    max_tokens=500,
)

In [ ]:
# response = llm.invoke("What is AI? Max 300 words.")
# print(response.content)

## **RAG IMPLEMENTATION with PDF data**

#### **Step 1: Extracting Text from PDF**

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

pdf_path = "./Docs/fabric-onelake.pdf"

loader = PyPDFLoader(pdf_path)
docs = loader.load()
docs

#### **STEP 1.1:(Optional) METADATA creation**

In [ ]:
x=0
for i in docs:
    i.metadata = {
        "source":"fabric-onelake.pdf",
        "developer":"Microsoft",
        "page":f"{x}"
        }
    x = x+1

#### **Step 2: CHUNKING**

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

chunks = splitter.split_documents(docs)
len(chunks)

In [ ]:
chunks[0].metadata

#### **STEP 3: Configure EMBEDDING MODEL**

In [3]:
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(
    model="qwen/qwen3-embedding-8b",
    api_key=api_key,
    base_url="https://openrouter.ai/api/v1",
    # IMPORTANT for OpenRouter
    check_embedding_ctx_length=False,
    # Recommended for OpenRouter-compatible providers
    # encoding_format="float",
)

#### **Step 4: Create and store embeddings in Vector store finally saving locally**

In [ ]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="./VectorStore/",
    
)

#----------behind the scenes-----------
# vectorstore=[]
# for doc in chunks:
#   vector = embedding_model.embed_documents([doc.page_content])
#   vectorstore.append(vector)

##### **Step 4.1: LOCAL VECTOR STORE**

In [4]:
from langchain_chroma import Chroma

vectorstore = Chroma(
    persist_directory="./VectorStore/",
    embedding_function=embedding_model
)

#### **Step 5: SEMANTIC SEARCH** 

In [8]:
context = vectorstore.similarity_search("Why do we need OneLake?",k=3)
context

[Document(id='03e20ee4-9e62-45d6-a1a6-57a7a4a5c9b9', metadata={'developer': 'Microsoft', 'source': 'fabric-onelake.pdf', 'page': '0'}, page_content="Microsoft OneLake documentation\nMicrosoft OneLake is Fabric's single, unified, logical data lake for the whole organization.\nOneLake comes automatically with every Fabric tenant with no infrastructure to manage.\nAbout OneLake\nｅ OVERVIEW\nWhat is OneLake?\nOneLake security\nOneLake catalog\nOneLake access and APIs\nＹ ARCHITECTURE\nOneLake patterns and foundational capabilities\n｀ DEPLOY\nImplement medallion lakehouse architecture\nｂ GET STARTED\nQuickstart: Get data in OneLake\nOneLake file explorer\nFind data in the OneLake catalog\nUse Iceberg tables in OneLake\nOneLake shortcuts\nｐ CONCEPT\nWhat are shortcuts?\nｂ GET STARTED"),
 Document(id='8121fa90-3cb9-4f90-810e-72b3fddacaeb', metadata={'source': 'fabric-onelake.pdf', 'developer': 'Microsoft', 'page': '267'}, page_content='Integrate OneLake with Azure Databricks\nLast updated on 0

#### **LETS TALK TO LLM FINALLY**

In [ ]:
response_context = llm.invoke(f"How is AI being used in research fields?? You can answer using the following context: {context}")
print(response_context.content)